# 🔭 Fink/LSST — Full Cutout Timeline Viewer

For a single `diaObjectId`, visualize the complete temporal sequence of cutouts
across all filters (u, g, r, i, z, y) and all observation epochs.

**Goal**: Understand how a Rubin/LSST alert is born — watch the transient grow
epoch by epoch in the difference images.

**Prerequisite**: Run `fink_download_full_cutouts.py` first to download the data:
```bash
python fink_download_full_cutouts.py --obj_id <diaObjectId>
```

**Key design choice**: Within a given filter, **all cutouts share the same color scale**,
so flux variations are directly comparable across epochs.

**Column naming reminder (LSST DPDD schema):**
- Prefix `r:` → `diaSource` table field (≠ spectral band `r`)
- Prefix `f:` → Fink-computed field
- `r:band` ∈ {`u`, `g`, `r`, `i`, `z`, `y`} — the 6 Rubin spectral bands

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
from astropy.visualization import ZScaleInterval
import warnings

warnings.filterwarnings("ignore")

%matplotlib inline
matplotlib.rcParams["figure.dpi"] = 120

# Import the shared library for light curve plots and color constants
from rubinlsstskyalerts.fink_tools import (
    BAND_COLORS,
    DARK_BG,
    PANEL_BG,
    TEXT_COL,
    MUTED_COL,
    ACCENT,
    HIGHLIGHT,
    BORDER,
    RUBIN_ZEROPOINT,
    flux_to_mag,
    plot_lightcurve_flux,
    plot_lightcurve_mag,
)

# Also import the download function so the notebook can trigger a download
from fink_download_full_cutouts import download_full_cutouts

print("✓ Imports OK")

---
## 1 · Select the diaObjectId and download its cutouts

Pick `OBJ_ID` from the `fink_alert_browser` notebook.
Set `RUN_DOWNLOAD = True` the first time, then `False` once data is on disk.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  USER PARAMETERS                                               ║
# ╚══════════════════════════════════════════════════════════════════╝

# diaObjectId chosen from fink_alert_browser.ipynb
OBJ_ID = 170032915988086813

# Download parameters
RUN_DOWNLOAD = True  # set to False once data is already on disk
SKIP_EXISTING = True  # skip diaSources whose .npy files already exist

# Output directory (auto-named by default)
OUTDIR = Path(f"fullcutouts_{OBJ_ID}")

# ─────────────────────────────────────────────────────────────────────────────
if RUN_DOWNLOAD:
    download_full_cutouts(OBJ_ID, outdir=OUTDIR, skip_existing=SKIP_EXISTING)
else:
    print(f"Skipping download — using existing data in {OUTDIR}")

print(f"\nFink portal : https://lsst.fink-portal.org/{OBJ_ID}")

In [ ]:
# ── Load manifest ─────────────────────────────────────────────────────────────
manifest = pd.read_parquet(OUTDIR / "manifest.parquet")

# Keep only successfully downloaded diaSources
manifest_ok = manifest[manifest["status"].isin(["ok", "skipped"])].copy()
manifest_ok = manifest_ok.sort_values("r:midpointMjdTai").reset_index(drop=True)

print(f"diaObjectId : {OBJ_ID}")
print(f"diaSources  : {len(manifest_ok)} available  ({len(manifest) - len(manifest_ok)} failed)")
print(
    f'MJD range   : {manifest_ok["r:midpointMjdTai"].min():.4f} → {manifest_ok["r:midpointMjdTai"].max():.4f}'
)
print(f"\nPer-band breakdown:")
print(
    manifest_ok.groupby("r:band")
    .agg(
        n_epochs=("r:diaSourceId", "count"),
        mjd_first=("r:midpointMjdTai", "min"),
        mjd_last=("r:midpointMjdTai", "max"),
        snr_mean=("r:snr", "mean"),
    )
    .to_string()
)
print(f"\nManifest columns: {list(manifest_ok.columns)}")

---
## 2 · Visualization parameters

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  VISUALIZATION PARAMETERS                                      ║
# ╚══════════════════════════════════════════════════════════════════╝

# 'triplet'   → show (Science, Template, Difference) for each epoch
# 'difference' → show only the Difference image for each epoch
DISPLAY_MODE = "triplet"

# Filters to display (None = all filters present in the data)
FILTERS = None  # e.g. ['g', 'r', 'i'] to restrict

# Color scale strategy within each filter:
# 'shared'  → same vmin/vmax for ALL epochs in a filter (recommended)
# 'zscale'  → independent ZScale per epoch
COLORSCALE = "shared"

# ZScale contrast parameter (lower = more dynamic range shown)
ZSCALE_CONTRAST = 0.25

# Max number of epochs to display per filter (None = all)
MAX_EPOCHS_PER_FILTER = None

# Figure width per cutout panel (inches)
PANEL_WIDTH = 2.8
PANEL_HEIGHT = 2.8

# Save figures to disk?
SAVE_FIGURES = False

# Colormap for Science and Template
CMAP_SCI = "afmhot"
CMAP_DIFF = "RdBu_r"  # diverging: blue=negative, red=positive

print("✓ Parameters set")
print(f"  display_mode : {DISPLAY_MODE}")
print(f"  colorscale   : {COLORSCALE}")
print(f'  filters      : {FILTERS or "all"}')

---
## 3 · Helper functions

In [ ]:
def load_cutout(row: pd.Series, kind: str) -> np.ndarray | None:
    """Load a single cutout array from disk."""
    path = Path(row[f"path_{kind}"])
    if not path.exists():
        return None
    return np.load(path)


def compute_shared_scale(
    rows: pd.DataFrame,
    kind: str,
    contrast: float = ZSCALE_CONTRAST,
) -> tuple[float, float]:
    """
    Compute a shared color scale (vmin, vmax) for all epochs of one
    filter and one cutout kind, using ZScale on the stacked array.

    Using a shared scale is crucial to compare flux levels across time:
    a rising transient will appear brighter in later epochs.
    """
    arrays = []
    for _, row in rows.iterrows():
        arr = load_cutout(row, kind)
        if arr is not None:
            arrays.append(arr.ravel())
    if not arrays:
        return 0.0, 1.0
    stacked = np.concatenate(arrays)
    try:
        return ZScaleInterval(contrast=contrast).get_limits(stacked)
    except Exception:
        return float(np.nanpercentile(stacked, 1)), float(np.nanpercentile(stacked, 99))


def _style_img_ax(ax, title="", subtitle=""):
    """Apply dark theme to an image axis."""
    ax.set_facecolor(PANEL_BG)
    for sp in ax.spines.values():
        sp.set_edgecolor(BORDER)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    if title:
        ax.set_title(title, color=ACCENT, fontsize=8, pad=3)
    if subtitle:
        ax.text(
            0.5,
            -0.02,
            subtitle,
            transform=ax.transAxes,
            ha="center",
            va="top",
            color=MUTED_COL,
            fontsize=6.5,
            fontfamily="monospace",
        )


print("✓ Helper functions defined")

---
## 4 · Light curve overview

In [ ]:
# Light curve (flux and mag) using the manifest as the light curve source
# The manifest contains ALL diaSources, so this is the most complete view.

fig, (ax_flux, ax_mag) = plt.subplots(1, 2, figsize=(14, 5), facecolor=DARK_BG)

t0 = manifest_ok["r:midpointMjdTai"].min()

for band in sorted(manifest_ok["r:band"].unique()):
    sub = manifest_ok[manifest_ok["r:band"] == band]
    t = sub["r:midpointMjdTai"].values - t0
    f = sub["r:psfFlux"].values
    fe = sub["r:psfFluxErr"].values
    col = BAND_COLORS.get(band, "gray")

    ax_flux.errorbar(t, f, yerr=fe, fmt="o", color=col, label=f"{band}", markersize=5, capsize=2, lw=1)

    mag, merr = flux_to_mag(f, fe)
    valid = np.isfinite(mag)
    if valid.sum() > 0:
        ax_mag.errorbar(
            t[valid],
            mag[valid],
            yerr=merr[valid],
            fmt="o",
            color=col,
            label=f"{band}",
            markersize=5,
            capsize=2,
            lw=1,
        )

for ax, ylabel, title in [
    (ax_flux, "psfFlux (nJy)", "Light curve — flux"),
    (ax_mag, "mag AB", "Light curve — mag AB"),
]:
    ax.set_facecolor(PANEL_BG)
    for sp in ax.spines.values():
        sp.set_edgecolor(BORDER)
    ax.tick_params(colors=MUTED_COL, labelsize=8)
    ax.set_xlabel("Δ MJD TAI (days)", color=MUTED_COL, fontsize=9)
    ax.set_ylabel(ylabel, color=MUTED_COL, fontsize=9)
    ax.set_title(title, color=ACCENT, fontsize=10)
    ax.legend(fontsize=8, facecolor=DARK_BG, labelcolor=TEXT_COL, edgecolor=BORDER)

ax_flux.axhline(0, color="#444", lw=0.8, ls="--")
ax_mag.invert_yaxis()

fig.suptitle(
    f"diaObjectId {OBJ_ID}  —  all bands  —  {len(manifest_ok)} diaSources",
    color=TEXT_COL,
    fontsize=11,
    fontfamily="monospace",
    y=1.01,
)
plt.tight_layout()

if SAVE_FIGURES:
    fig.savefig(OUTDIR / f"{OBJ_ID}_lightcurve.png", dpi=150, bbox_inches="tight", facecolor=DARK_BG)
plt.show()

---
## 5 · Cutout timeline — one figure per filter

Each row = one observation epoch (diaSource), sorted by MJD.  
Color scale is **shared across all epochs within a filter** → flux variations are directly visible.

In [ ]:
# Determine which filters to display
filters_in_data = sorted(manifest_ok["r:band"].unique())
filters_to_show = FILTERS if FILTERS else filters_in_data
filters_to_show = [b for b in filters_to_show if b in filters_in_data]

print(f"Filters present in data : {filters_in_data}")
print(f"Filters to display      : {filters_to_show}")
print(f"Display mode            : {DISPLAY_MODE}")
print(f"Color scale             : {COLORSCALE}")

In [ ]:
# ── Main timeline plot — one figure per filter ────────────────────────────────

# Number of cutout columns per epoch
n_cutout_cols = 3 if DISPLAY_MODE == "triplet" else 1
kinds_to_show = ["Science", "Template", "Difference"] if DISPLAY_MODE == "triplet" else ["Difference"]

t0_global = manifest_ok["r:midpointMjdTai"].min()  # reference epoch for all filters

for band in filters_to_show:
    band_rows = manifest_ok[manifest_ok["r:band"] == band].copy()
    band_rows = band_rows.sort_values("r:midpointMjdTai").reset_index(drop=True)

    if MAX_EPOCHS_PER_FILTER is not None:
        band_rows = band_rows.head(MAX_EPOCHS_PER_FILTER)

    n_epochs = len(band_rows)
    if n_epochs == 0:
        continue

    print(f'\n{'='*60}')
    print(f"Filter {band}  —  {n_epochs} epochs")
    print(f'{'='*60}')

    # ── Compute shared color scale across all epochs ──────────────────────────
    scales = {}  # kind → (vmin, vmax)
    if COLORSCALE == "shared":
        for kind in kinds_to_show:
            vmin, vmax = compute_shared_scale(band_rows, kind, ZSCALE_CONTRAST)
            if kind == "Difference":
                # symmetric around 0 for diverging colormap
                absmax = max(abs(vmin), abs(vmax))
                vmin, vmax = -absmax, absmax
            scales[kind] = (vmin, vmax)
            print(f"  Shared scale {kind:12s} : vmin={vmin:.1f}  vmax={vmax:.1f}")

    # ── Build figure ──────────────────────────────────────────────────────────
    # Layout: rows = epochs, cols = cutout kinds + 1 light curve strip
    # Light curve strip is a narrow column on the left showing flux vs MJD
    # with a horizontal line at the current epoch.

    lc_width = 1.8  # inches for the light curve column
    fig_width = lc_width + n_cutout_cols * PANEL_WIDTH
    fig_height = n_epochs * PANEL_HEIGHT

    fig = plt.figure(figsize=(fig_width, fig_height), facecolor=DARK_BG)
    fig.suptitle(
        f"diaObjectId {OBJ_ID}  —  filter {band}  —  {n_epochs} epochs\n"
        f"Color scale: {COLORSCALE}  |  Mode: {DISPLAY_MODE}",
        color=TEXT_COL,
        fontsize=10,
        fontfamily="monospace",
        y=1.005,
    )

    # GridSpec: n_epochs rows × (1 LC column + n_cutout_cols cutout columns)
    gs = gridspec.GridSpec(
        n_epochs,
        1 + n_cutout_cols,
        figure=fig,
        width_ratios=[lc_width / PANEL_WIDTH] + [1] * n_cutout_cols,
        hspace=0.05,
        wspace=0.05,
        left=0.01,
        right=0.99,
        top=0.97,
        bottom=0.02,
    )

    # Light curve data for this band (for the LC strip)
    lc_t = band_rows["r:midpointMjdTai"].values - t0_global
    lc_f = band_rows["r:psfFlux"].values
    lc_fe = band_rows["r:psfFluxErr"].values
    band_color = BAND_COLORS.get(band, "gray")

    for epoch_idx, (_, row) in enumerate(band_rows.iterrows()):
        mjd = row["r:midpointMjdTai"]
        dt = mjd - t0_global
        snr = row.get("r:snr", float("nan"))
        snn = row.get("f:clf_snnSnVsOthers_score", float("nan"))
        src_id = int(row["r:diaSourceId"])

        # ── Light curve strip (col 0) ─────────────────────────────────────────
        ax_lc = fig.add_subplot(gs[epoch_idx, 0])
        ax_lc.set_facecolor(PANEL_BG)
        ax_lc.errorbar(
            lc_t, lc_f, yerr=lc_fe, fmt="o", color=band_color, markersize=3, capsize=1, lw=0.8, alpha=0.6
        )
        # Highlight current epoch
        ax_lc.axvline(dt, color=HIGHLIGHT, lw=1.5, ls="-", alpha=0.9)
        ax_lc.scatter([dt], [row["r:psfFlux"]], color=HIGHLIGHT, s=30, zorder=5)
        ax_lc.axhline(0, color="#444", lw=0.5, ls="--")
        ax_lc.set_xlim(lc_t.min() - 0.5, lc_t.max() + 0.5)
        for sp in ax_lc.spines.values():
            sp.set_edgecolor(BORDER)
        ax_lc.tick_params(colors=MUTED_COL, labelsize=6)
        # Only show x-label on last row
        if epoch_idx < n_epochs - 1:
            ax_lc.set_xticklabels([])
        else:
            ax_lc.set_xlabel("Δ MJD", color=MUTED_COL, fontsize=7)
        # Metadata label on y axis
        ax_lc.set_ylabel(
            f"MJD+{dt:.2f}\nSNR={snr:.1f}\nSNN={snn:.2f}",
            color=MUTED_COL,
            fontsize=5.5,
            rotation=0,
            labelpad=48,
            va="center",
        )

        # ── Cutout panels ─────────────────────────────────────────────────────
        for col_idx, kind in enumerate(kinds_to_show):
            ax = fig.add_subplot(gs[epoch_idx, col_idx + 1])
            ax.set_facecolor(PANEL_BG)

            arr = load_cutout(row, kind)
            if arr is not None:
                cmap = CMAP_DIFF if kind == "Difference" else CMAP_SCI

                if COLORSCALE == "shared":
                    vmin, vmax = scales[kind]
                else:
                    # Independent ZScale per epoch
                    try:
                        vmin, vmax = ZScaleInterval(contrast=ZSCALE_CONTRAST).get_limits(arr)
                    except Exception:
                        vmin, vmax = np.nanpercentile(arr, [1, 99])
                    if kind == "Difference":
                        absmax = max(abs(vmin), abs(vmax))
                        vmin, vmax = -absmax, absmax

                ax.imshow(arr, origin="lower", cmap=cmap, vmin=vmin, vmax=vmax, interpolation="nearest")

                # Crosshair at center
                H, W = arr.shape
                ax.plot(W / 2, H / 2, "+", color=HIGHLIGHT, ms=10, mew=1.2)

                # SNR annotation on Difference image
                if kind == "Difference":
                    ax.text(
                        0.02,
                        0.97,
                        f"SNR={snr:.1f}",
                        transform=ax.transAxes,
                        color=HIGHLIGHT,
                        fontsize=6,
                        va="top",
                        fontfamily="monospace",
                        bbox=dict(facecolor=DARK_BG, alpha=0.6, edgecolor="none"),
                    )
            else:
                ax.text(
                    0.5,
                    0.5,
                    "N/A",
                    transform=ax.transAxes,
                    ha="center",
                    va="center",
                    color=MUTED_COL,
                    fontsize=8,
                )

            for sp in ax.spines.values():
                sp.set_edgecolor(BORDER)
            ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

            # Column header only on first row
            if epoch_idx == 0:
                ax.set_title(kind, color=ACCENT, fontsize=9, pad=4)

    plt.tight_layout(rect=[0, 0, 1, 0.998])

    if SAVE_FIGURES:
        fname = OUTDIR / f"{OBJ_ID}_timeline_{band}_{DISPLAY_MODE}.png"
        fig.savefig(fname, dpi=150, bbox_inches="tight", facecolor=DARK_BG)
        print(f"  ✓ Saved: {fname}")

    plt.show()
    plt.close(fig)

---
## 6 · Difference images only — all filters on one figure

Compact view: rows = filters, columns = epochs (sorted by MJD).  
Color scale is shared per row (per filter).

In [ ]:
# Compact multi-filter grid: rows = filters, cols = epochs
# Each cell = Difference image. Color scale shared per row (per filter).

# Find the maximum number of epochs across filters
max_epochs = max(len(manifest_ok[manifest_ok["r:band"] == b]) for b in filters_to_show)

n_rows = len(filters_to_show)
n_cols = max_epochs if MAX_EPOCHS_PER_FILTER is None else min(max_epochs, MAX_EPOCHS_PER_FILTER)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(n_cols * 2.0, n_rows * 2.2),
    facecolor=DARK_BG,
    squeeze=False,
)
fig.suptitle(
    f"Difference images — diaObjectId {OBJ_ID}\n"
    f"rows=filters, cols=epochs (MJD increasing →), shared color scale per filter",
    color=TEXT_COL,
    fontsize=10,
    fontfamily="monospace",
    y=1.02,
)

t0_global = manifest_ok["r:midpointMjdTai"].min()

for row_idx, band in enumerate(filters_to_show):
    band_rows = manifest_ok[manifest_ok["r:band"] == band].sort_values("r:midpointMjdTai")
    if MAX_EPOCHS_PER_FILTER:
        band_rows = band_rows.head(MAX_EPOCHS_PER_FILTER)

    # Shared scale for this filter's Difference images
    vmin, vmax = compute_shared_scale(band_rows, "Difference", ZSCALE_CONTRAST)
    absmax = max(abs(vmin), abs(vmax))
    vmin, vmax = -absmax, absmax

    for col_idx in range(n_cols):
        ax = axes[row_idx, col_idx]
        ax.set_facecolor(PANEL_BG)
        for sp in ax.spines.values():
            sp.set_edgecolor(BORDER)
        ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

        if col_idx < len(band_rows):
            epoch_row = band_rows.iloc[col_idx]
            arr = load_cutout(epoch_row, "Difference")
            dt = epoch_row["r:midpointMjdTai"] - t0_global
            snr = epoch_row.get("r:snr", float("nan"))

            if arr is not None:
                H, W = arr.shape
                ax.imshow(arr, origin="lower", cmap=CMAP_DIFF, vmin=vmin, vmax=vmax, interpolation="nearest")
                ax.plot(W / 2, H / 2, "+", color=HIGHLIGHT, ms=8, mew=1)
                ax.text(
                    0.03,
                    0.97,
                    f"Δt={dt:.1f}d\nSNR={snr:.1f}",
                    transform=ax.transAxes,
                    color=HIGHLIGHT,
                    fontsize=5.5,
                    va="top",
                    fontfamily="monospace",
                    bbox=dict(facecolor=DARK_BG, alpha=0.6, edgecolor="none"),
                )
            else:
                ax.text(
                    0.5,
                    0.5,
                    "N/A",
                    transform=ax.transAxes,
                    ha="center",
                    va="center",
                    color=MUTED_COL,
                    fontsize=7,
                )
        else:
            # No epoch for this column
            ax.set_visible(False)

        # Filter label on left column only
        if col_idx == 0:
            ax.set_ylabel(f"band {band}", color=BAND_COLORS.get(band, "gray"), fontsize=10, fontweight="bold")

        # Epoch index on top row only
        if row_idx == 0 and col_idx < len(band_rows):
            ax.set_title(f"ep {col_idx+1}", color=MUTED_COL, fontsize=7, pad=2)

plt.tight_layout()

if SAVE_FIGURES:
    fname = OUTDIR / f"{OBJ_ID}_diff_grid.png"
    fig.savefig(fname, dpi=150, bbox_inches="tight", facecolor=DARK_BG)
    print(f"✓ Saved: {fname}")

plt.show()

---
## 7 · Per-filter Difference image animation (static mosaic)

For a single chosen filter, show the Difference images in temporal order
with the light curve underneath. Useful for understanding how the alert is born.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  Choose the filter for the detailed mosaic                     ║
# ╚══════════════════════════════════════════════════════════════════╝
MOSAIC_BAND = filters_to_show[0]  # ← change to 'g', 'r', 'i', etc.

band_rows = manifest_ok[manifest_ok["r:band"] == MOSAIC_BAND].sort_values("r:midpointMjdTai")
n_epochs = len(band_rows)
t0 = manifest_ok["r:midpointMjdTai"].min()

# Shared scales for this band
sci_vmin, sci_vmax = compute_shared_scale(band_rows, "Science", ZSCALE_CONTRAST)
tmpl_vmin, tmpl_vmax = compute_shared_scale(band_rows, "Template", ZSCALE_CONTRAST)
diff_vmin, diff_vmax = compute_shared_scale(band_rows, "Difference", ZSCALE_CONTRAST)
diff_abs = max(abs(diff_vmin), abs(diff_vmax))
diff_vmin, diff_vmax = -diff_abs, diff_abs

print(f"Band {MOSAIC_BAND}  —  {n_epochs} epochs")
print(f"Science  scale : [{sci_vmin:.1f}, {sci_vmax:.1f}]")
print(f"Template scale : [{tmpl_vmin:.1f}, {tmpl_vmax:.1f}]")
print(f"Diff     scale : [{diff_vmin:.1f}, {diff_vmax:.1f}]  (symmetric)")

# Figure: top row = Difference mosaique, bottom row = light curve
fig = plt.figure(figsize=(max(n_epochs * 2.0, 8), 6), facecolor=DARK_BG)
gs_main = gridspec.GridSpec(
    2, 1, figure=fig, height_ratios=[3, 1], hspace=0.12, left=0.04, right=0.98, top=0.93, bottom=0.06
)

# Top: Difference cutouts
gs_top = gridspec.GridSpecFromSubplotSpec(1, n_epochs, subplot_spec=gs_main[0], wspace=0.04)

for col_idx, (_, row) in enumerate(band_rows.iterrows()):
    ax = fig.add_subplot(gs_top[0, col_idx])
    arr = load_cutout(row, "Difference")
    dt = row["r:midpointMjdTai"] - t0
    snr = row.get("r:snr", float("nan"))

    ax.set_facecolor(PANEL_BG)
    if arr is not None:
        H, W = arr.shape
        ax.imshow(
            arr, origin="lower", cmap=CMAP_DIFF, vmin=diff_vmin, vmax=diff_vmax, interpolation="nearest"
        )
        ax.plot(W / 2, H / 2, "+", color=HIGHLIGHT, ms=10, mew=1.3)
        ax.text(
            0.05,
            0.95,
            f"SNR\n{snr:.1f}",
            transform=ax.transAxes,
            color=HIGHLIGHT,
            fontsize=6,
            va="top",
            fontfamily="monospace",
            bbox=dict(facecolor=DARK_BG, alpha=0.55, edgecolor="none"),
        )
    ax.set_title(f"Δt={dt:.2f}d", color=MUTED_COL, fontsize=7, pad=2)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    for sp in ax.spines.values():
        sp.set_edgecolor(BORDER)

# Bottom: light curve for this band
ax_lc = fig.add_subplot(gs_main[1])
ax_lc.set_facecolor(PANEL_BG)
t_all = band_rows["r:midpointMjdTai"].values - t0
f_all = band_rows["r:psfFlux"].values
fe_all = band_rows["r:psfFluxErr"].values
col = BAND_COLORS.get(MOSAIC_BAND, "gray")

ax_lc.errorbar(t_all, f_all, yerr=fe_all, fmt="o-", color=col, markersize=5, capsize=2, lw=1)
ax_lc.axhline(0, color="#444", lw=0.7, ls="--")
ax_lc.fill_between(t_all, f_all - fe_all, f_all + fe_all, color=col, alpha=0.15)
# Vertical lines at each epoch
for dt in t_all:
    ax_lc.axvline(dt, color=HIGHLIGHT, lw=0.5, ls=":", alpha=0.5)

ax_lc.set_xlabel("Δ MJD TAI (days)", color=MUTED_COL, fontsize=8)
ax_lc.set_ylabel("psfFlux (nJy)", color=MUTED_COL, fontsize=8)
ax_lc.tick_params(colors=MUTED_COL, labelsize=7)
ax_lc.set_xlim(t_all.min() - 0.3, t_all.max() + 0.3)
for sp in ax_lc.spines.values():
    sp.set_edgecolor(BORDER)

fig.suptitle(
    f"Difference image birth sequence — diaObjectId {OBJ_ID}  |  filter {MOSAIC_BAND}\n"
    f"Shared color scale: [{diff_vmin:.0f}, {diff_vmax:.0f}] ADU  "
    f"(RdBu_r: blue=negative, red=positive)",
    color=TEXT_COL,
    fontsize=10,
    fontfamily="monospace",
)

if SAVE_FIGURES:
    fname = OUTDIR / f"{OBJ_ID}_birth_{MOSAIC_BAND}.png"
    fig.savefig(fname, dpi=150, bbox_inches="tight", facecolor=DARK_BG)
    print(f"✓ Saved: {fname}")

plt.show()

---
## 8 · Dataset preparation for machine learning

Build a labeled array dataset from all available cutouts.
Each sample is a `(3, H, W)` array (Science, Template, Difference).
The target can be the SNN score (regression) or a binary class (classification).

In [ ]:
# ── Build the ML dataset from all downloaded cutouts ─────────────────────────

# Standard cutout size — pad or crop to this size if needed
TARGET_H, TARGET_W = 30, 30


def resize_cutout(arr: np.ndarray, h: int, w: int) -> np.ndarray:
    """
    Center-crop or zero-pad a 2D array to (h, w).
    Rubin cutouts are typically 30×30 but can vary.
    """
    ah, aw = arr.shape
    # Crop if larger
    if ah > h:
        start = (ah - h) // 2
        arr = arr[start : start + h, :]
    if aw > w:
        start = (aw - w) // 2
        arr = arr[:, start : start + w]
    # Pad if smaller
    pad_h = max(0, h - arr.shape[0])
    pad_w = max(0, w - arr.shape[1])
    arr = np.pad(
        arr,
        ((pad_h // 2, pad_h - pad_h // 2), (pad_w // 2, pad_w - pad_w // 2)),
        mode="constant",
        constant_values=0,
    )
    return arr


samples_X = []  # (3, H, W) float32 arrays
samples_meta = []  # dict with metadata per sample

for _, row in manifest_ok.iterrows():
    sci = load_cutout(row, "Science")
    tmpl = load_cutout(row, "Template")
    diff = load_cutout(row, "Difference")

    if sci is None or tmpl is None or diff is None:
        continue

    # Resize all 3 planes to the same shape
    sci = resize_cutout(sci, TARGET_H, TARGET_W)
    tmpl = resize_cutout(tmpl, TARGET_H, TARGET_W)
    diff = resize_cutout(diff, TARGET_H, TARGET_W)

    sample = np.stack([sci, tmpl, diff], axis=0)  # (3, H, W)
    samples_X.append(sample)

    samples_meta.append(
        {
            "r:diaObjectId": row["r:diaObjectId"],
            "r:diaSourceId": row["r:diaSourceId"],
            "r:band": row["r:band"],
            "r:midpointMjdTai": row["r:midpointMjdTai"],
            "r:snr": row.get("r:snr"),
            "f:clf_snnSnVsOthers_score": row.get("f:clf_snnSnVsOthers_score"),
            # Binary label: SNN score >= 0.5 → potential SN
            "label_sn": int(float(row.get("f:clf_snnSnVsOthers_score", 0) or 0) >= 0.5),
        }
    )

X = np.array(samples_X, dtype=np.float32)  # (N, 3, H, W)
df_meta = pd.DataFrame(samples_meta)
y_snn = df_meta["f:clf_snnSnVsOthers_score"].values.astype(np.float32)
y_binary = df_meta["label_sn"].values

print(f"ML dataset ready:")
print(f"  X shape      : {X.shape}  dtype={X.dtype}")
print(f"  Channels     : [0]=Science  [1]=Template  [2]=Difference")
print(f"  y_snn range  : [{y_snn.min():.3f}, {y_snn.max():.3f}]")
print(f"  y_binary     : {np.bincount(y_binary)}  (0=not SN, 1=potential SN)")
print(f'  Bands present: {sorted(df_meta["r:band"].unique())}')

In [ ]:
# ── Save the ML dataset ───────────────────────────────────────────────────────
np.save(OUTDIR / "X_cutouts.npy", X)
df_meta.to_parquet(OUTDIR / "y_meta.parquet", index=False)
df_meta.to_csv(OUTDIR / "y_meta.csv", index=False)

print(f"Saved:")
print(f"  {OUTDIR}/X_cutouts.npy    shape={X.shape}")
print(f"  {OUTDIR}/y_meta.parquet")
print(f"  {OUTDIR}/y_meta.csv")

In [ ]:
# ── Quick sanity check: pixel statistics per channel ─────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(12, 4), facecolor=DARK_BG)
channel_names = ["Science", "Template", "Difference"]
cmaps = ["afmhot", "afmhot", "RdBu_r"]

for ch, (ax, name, cmap) in enumerate(zip(axes, channel_names, cmaps)):
    ax.set_facecolor(PANEL_BG)
    # Show mean image across all samples for this channel
    mean_img = X[:, ch, :, :].mean(axis=0)
    vmin, vmax = np.nanpercentile(mean_img, [5, 95])
    if name == "Difference":
        absmax = max(abs(vmin), abs(vmax))
        vmin, vmax = -absmax, absmax
    im = ax.imshow(mean_img, origin="lower", cmap=cmap, vmin=vmin, vmax=vmax, interpolation="nearest")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04).ax.tick_params(labelsize=6, colors=MUTED_COL)
    # Crosshair
    H, W = mean_img.shape
    ax.plot(W / 2, H / 2, "+", color=HIGHLIGHT, ms=12, mew=1.5)
    ax.set_title(f"Mean {name}  (N={len(X)})", color=ACCENT, fontsize=10)
    ax.tick_params(colors=MUTED_COL, labelsize=7)
    for sp in ax.spines.values():
        sp.set_edgecolor(BORDER)

fig.suptitle(
    f"Mean cutout per channel — diaObjectId {OBJ_ID}\n" f"Averaged over {len(X)} diaSources across all bands",
    color=TEXT_COL,
    fontsize=10,
    fontfamily="monospace",
)
plt.tight_layout()
plt.show()

print("\nPixel statistics per channel:")
for ch, name in enumerate(channel_names):
    data = X[:, ch, :, :].ravel()
    print(
        f"  {name:12s} : mean={data.mean():.1f}  std={data.std():.1f}  "
        f"min={data.min():.1f}  max={data.max():.1f}"
    )